# 정수 인코딩 (Integer Encoding) 및 패딩 (Padding)

## 정수 인코딩 (Integer Encoding)

정수 인코딩은 텍스트 데이터를 컴퓨터가 처리할 수 있도록 단어마다 고유한 정수로 변환하는 방식이다.

컴퓨터는 원래 텍스트를 직접 이해하지 못하고, 숫자 형태의 데이터만 처리 가능하다. 따라서 텍스트를 숫자로 변환하면 머신러닝 모델이 이를 입력값으로 사용할 수 있다.

정수 인코딩의 핵심은 인덱스 부여 기준이다. 일반적으로 단어가 텍스트에서 얼마나 자주 나타나는지에 따라 인덱스를 부여한다. 즉, 빈도가 높은 단어에는 낮은 숫자를 부여하고, 빈도가 낮은 단어에는 높은 숫자를 부여하는 방식이다. 이렇게 하면 메모리를 효율적으로 사용할 수 있고 계산 속도도 빨라진다.

실제로는 모든 단어를 다 인코딩하지 않고 상위 5,000개 단어 정도로 제한하는 경우가 많다. 왜냐하면 자주 나타나지 않는 단어들은 노이즈가 될 수 있고, 불필요한 자원 소비를 줄일 수 있으며, 궁극적으로 모델의 성능을 향상시킬 수 있기 때문이다.

예를 들어, "안녕은 1번", "밥은 2번", "점심은 3번" 이런 식으로 각 단어에 고유한 숫자를 부여하는 것이다.

## 패딩 (Padding)

자연어 처리에서 각 문장(또는 문서)은 길이가 다를 수 있다.

그러나 언어모델은 고정된 길이의 데이터를 효율적으로 처리하기 때문에, 모든 문장의 길이를 동일하게 맞춰주는 작업이 필요하다.

이 작업을 **패딩(padding)** 이라고 하며, **제로 패딩(zero padding)** 은 0으로 패딩하는 방법을 의미한다.

1. 일관된 입력 형식: 다양한 길이의 문장을 고정된 길이로 맞춰 모델이 일관된 형식으로 데이터를 학습할 수 있다.
2. 병렬 연산 최적화: 고정된 길이 덕분에 GPU 병렬 연산을 효율적으로 수행할 수 있다.
3. 유연한 데이터 처리: 텍스트 데이터의 길이를 효과적으로 조정하여 데이터 손실 없이 다양한 길이의 문장을 모델에 전달할 수 있다.


In [2]:
# 어린왕자 샘플 텍스트
raw_text = """The Little Prince, written by Antoine de Saint-Exupéry, is a poetic tale about a young prince who travels from his home planet to Earth. The story begins with a pilot stranded in the Sahara Desert after his plane crashes. While trying to fix his plane, he meets a mysterious young boy, the Little Prince.

The Little Prince comes from a small asteroid called B-612, where he lives alone with a rose that he loves deeply. He recounts his journey to the pilot, describing his visits to several other planets. Each planet is inhabited by a different character, such as a king, a vain man, a drunkard, a businessman, a geographer, and a fox. Through these encounters, the Prince learns valuable lessons about love, responsibility, and the nature of adult behavior.

On Earth, the Little Prince meets various creatures, including a fox, who teaches him about relationships and the importance of taming, which means building ties with others. The fox's famous line, "You become responsible, forever, for what you have tamed," resonates with the Prince's feelings for his rose.

Ultimately, the Little Prince realizes that the essence of life is often invisible and can only be seen with the heart. After sharing his wisdom with the pilot, he prepares to return to his asteroid and his beloved rose. The story concludes with the pilot reflecting on the lessons learned from the Little Prince and the enduring impact of their friendship.

The narrative is a beautifully simple yet profound exploration of love, loss, and the importance of seeing beyond the surface of things."""


## 토큰화

In [3]:
# 문장 토큰화 및 단어 토큰화
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

sentences = sent_tokenize(raw_text)
en_stopwords = stopwords.words('english')

# 단어 사전 생성
vocab = {}
preprocessed_sentences = []

for sentence in sentences:
    sentence = sentence.lower()
    tokens = word_tokenize(sentence)
    # 불용어, 단어 길이가 3개 미만 필터링
    tokens = [token for token in tokens if token not in en_stopwords and len(token) > 2]
    preprocessed_sentences.append(tokens)

    # 단어 빈도 사전 추가
    for token in tokens:
        if token not in vocab:
            vocab[token] = 0        # 초기값 설정
        vocab[token] += 1           # 있을 경우 값 증가

In [4]:
for i, sent in enumerate(preprocessed_sentences):
    print(f'{i} : {sent}')

0 : ['little', 'prince', 'written', 'antoine', 'saint-exupéry', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth']
1 : ['story', 'begins', 'pilot', 'stranded', 'sahara', 'desert', 'plane', 'crashes']
2 : ['trying', 'fix', 'plane', 'meets', 'mysterious', 'young', 'boy', 'little', 'prince']
3 : ['little', 'prince', 'comes', 'small', 'asteroid', 'called', 'b-612', 'lives', 'alone', 'rose', 'loves', 'deeply']
4 : ['recounts', 'journey', 'pilot', 'describing', 'visits', 'several', 'planets']
5 : ['planet', 'inhabited', 'different', 'character', 'king', 'vain', 'man', 'drunkard', 'businessman', 'geographer', 'fox']
6 : ['encounters', 'prince', 'learns', 'valuable', 'lessons', 'love', 'responsibility', 'nature', 'adult', 'behavior']
7 : ['earth', 'little', 'prince', 'meets', 'various', 'creatures', 'including', 'fox', 'teaches', 'relationships', 'importance', 'taming', 'means', 'building', 'ties', 'others']
8 : ['fox', 'famous', 'line', 'become', 'responsible', 'foreve

In [5]:
print('단어빈도사전수 :',len(vocab))
print(vocab)

단어빈도사전수 : 109
{'little': 6, 'prince': 9, 'written': 1, 'antoine': 1, 'saint-exupéry': 1, 'poetic': 1, 'tale': 1, 'young': 2, 'travels': 1, 'home': 1, 'planet': 2, 'earth': 2, 'story': 2, 'begins': 1, 'pilot': 4, 'stranded': 1, 'sahara': 1, 'desert': 1, 'plane': 2, 'crashes': 1, 'trying': 1, 'fix': 1, 'meets': 2, 'mysterious': 1, 'boy': 1, 'comes': 1, 'small': 1, 'asteroid': 2, 'called': 1, 'b-612': 1, 'lives': 1, 'alone': 1, 'rose': 3, 'loves': 1, 'deeply': 1, 'recounts': 1, 'journey': 1, 'describing': 1, 'visits': 1, 'several': 1, 'planets': 1, 'inhabited': 1, 'different': 1, 'character': 1, 'king': 1, 'vain': 1, 'man': 1, 'drunkard': 1, 'businessman': 1, 'geographer': 1, 'fox': 3, 'encounters': 1, 'learns': 1, 'valuable': 1, 'lessons': 2, 'love': 2, 'responsibility': 1, 'nature': 1, 'adult': 1, 'behavior': 1, 'various': 1, 'creatures': 1, 'including': 1, 'teaches': 1, 'relationships': 1, 'importance': 2, 'taming': 1, 'means': 1, 'building': 1, 'ties': 1, 'others': 1, 'famous': 1, 'li

In [6]:
# 빈도 수 역순으로 정렬
import pandas as pd

vocab_sorted = sorted(vocab.items(), key=lambda x:x[1], reverse=True)
vocab_df = pd.DataFrame(vocab_sorted,columns=['word','freq'])
vocab_df

,word,freq
0,prince,9
1,little,6
2,pilot,4
3,rose,3
4,fox,3
...,...,...
104,loss,1
105,seeing,1
106,beyond,1
107,surface,1


In [7]:
# word_to_index_dict (key:word, value:index)
# index_to_word_dict (key:index, value:word)

# 0번 padding 문자, 1번 oov문자 (out of vocabulary-unkown)
word_to_index = {word[0]: i+2 for i,word in enumerate(vocab_sorted)}

# 빈도수가 낮은 단어 제외 (전체 개수 중 상위 n개 단어 사용)
vocab_size = 30
word_to_index = {word:index for word, index in word_to_index.items() if index <= vocab_size}
word_to_index

{'prince': 2,
 'little': 3,
 'pilot': 4,
 'rose': 5,
 'fox': 6,
 'young': 7,
 'planet': 8,
 'earth': 9,
 'story': 10,
 'plane': 11,
 'meets': 12,
 'asteroid': 13,
 'lessons': 14,
 'love': 15,
 'importance': 16,
 'written': 17,
 'antoine': 18,
 'saint-exupéry': 19,
 'poetic': 20,
 'tale': 21,
 'travels': 22,
 'home': 23,
 'begins': 24,
 'stranded': 25,
 'sahara': 26,
 'desert': 27,
 'crashes': 28,
 'trying': 29,
 'fix': 30}

In [8]:
# oov문자 추가
word_to_index['OOV'] = 1
word_to_index

{'prince': 2,
 'little': 3,
 'pilot': 4,
 'rose': 5,
 'fox': 6,
 'young': 7,
 'planet': 8,
 'earth': 9,
 'story': 10,
 'plane': 11,
 'meets': 12,
 'asteroid': 13,
 'lessons': 14,
 'love': 15,
 'importance': 16,
 'written': 17,
 'antoine': 18,
 'saint-exupéry': 19,
 'poetic': 20,
 'tale': 21,
 'travels': 22,
 'home': 23,
 'begins': 24,
 'stranded': 25,
 'sahara': 26,
 'desert': 27,
 'crashes': 28,
 'trying': 29,
 'fix': 30,
 'OOV': 1}

In [9]:
# index_to_word
index_to_word = {index: word for word, index in  word_to_index.items()}
index_to_word

{2: 'prince',
 3: 'little',
 4: 'pilot',
 5: 'rose',
 6: 'fox',
 7: 'young',
 8: 'planet',
 9: 'earth',
 10: 'story',
 11: 'plane',
 12: 'meets',
 13: 'asteroid',
 14: 'lessons',
 15: 'love',
 16: 'importance',
 17: 'written',
 18: 'antoine',
 19: 'saint-exupéry',
 20: 'poetic',
 21: 'tale',
 22: 'travels',
 23: 'home',
 24: 'begins',
 25: 'stranded',
 26: 'sahara',
 27: 'desert',
 28: 'crashes',
 29: 'trying',
 30: 'fix',
 1: 'OOV'}

## 정수 인코딩

In [10]:
# 문장별로 단어를 정수 인코딩
encoded_sentences = []
oov = word_to_index['OOV']  # 1

for sent in preprocessed_sentences:
    encoded_sent = [word_to_index.get(word,oov) for word in sent]
    encoded_sentences.append(encoded_sent)
    print(sent)
    print(encoded_sent)
    print()

['little', 'prince', 'written', 'antoine', 'saint-exupéry', 'poetic', 'tale', 'young', 'prince', 'travels', 'home', 'planet', 'earth']
[3, 2, 17, 18, 19, 20, 21, 7, 2, 22, 23, 8, 9]

['story', 'begins', 'pilot', 'stranded', 'sahara', 'desert', 'plane', 'crashes']
[10, 24, 4, 25, 26, 27, 11, 28]

['trying', 'fix', 'plane', 'meets', 'mysterious', 'young', 'boy', 'little', 'prince']
[29, 30, 11, 12, 1, 7, 1, 3, 2]

['little', 'prince', 'comes', 'small', 'asteroid', 'called', 'b-612', 'lives', 'alone', 'rose', 'loves', 'deeply']
[3, 2, 1, 1, 13, 1, 1, 1, 1, 5, 1, 1]

['recounts', 'journey', 'pilot', 'describing', 'visits', 'several', 'planets']
[1, 1, 4, 1, 1, 1, 1]

['planet', 'inhabited', 'different', 'character', 'king', 'vain', 'man', 'drunkard', 'businessman', 'geographer', 'fox']
[8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6]

['encounters', 'prince', 'learns', 'valuable', 'lessons', 'love', 'responsibility', 'nature', 'adult', 'behavior']
[1, 2, 1, 1, 14, 15, 1, 1, 1, 1]

['earth', 'little', 'pr

In [11]:
encoded_sentences

[[3, 2, 17, 18, 19, 20, 21, 7, 2, 22, 23, 8, 9],
 [10, 24, 4, 25, 26, 27, 11, 28],
 [29, 30, 11, 12, 1, 7, 1, 3, 2],
 [3, 2, 1, 1, 13, 1, 1, 1, 1, 5, 1, 1],
 [1, 1, 4, 1, 1, 1, 1],
 [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6],
 [1, 2, 1, 1, 14, 15, 1, 1, 1, 1],
 [9, 3, 2, 12, 1, 1, 1, 6, 1, 1, 16, 1, 1, 1, 1, 1],
 [6, 1, 1, 1, 1, 1, 1, 1, 2, 1, 5],
 [1, 3, 2, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 4, 1, 1, 13, 1, 5],
 [10, 1, 4, 1, 14, 1, 3, 2, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 15, 1, 16, 1, 1, 1, 1]]

## 패딩 처리
문장별 길이가 모두 다르기 때문에 이를 통일해야 한다.

In [12]:
import numpy as np

# 문장 길이 고정
max_len = max(len(sent) for sent in encoded_sentences)

batch_size = len(encoded_sentences)

padded_sentences = np.zeros((batch_size,max_len),dtype=int)

for i, sent in enumerate(encoded_sentences):
    padded_sentences[i, :len(sent)] = sent

pd.DataFrame(padded_sentences)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,3,2,17,18,19,20,21,7,2,22,23,8,9,0,0,0
1,10,24,4,25,26,27,11,28,0,0,0,0,0,0,0,0
2,29,30,11,12,1,7,1,3,2,0,0,0,0,0,0,0
3,3,2,1,1,13,1,1,1,1,5,1,1,0,0,0,0
4,1,1,4,1,1,1,1,0,0,0,0,0,0,0,0,0
5,8,1,1,1,1,1,1,1,1,1,6,0,0,0,0,0
6,1,2,1,1,14,15,1,1,1,1,0,0,0,0,0,0
7,9,3,2,12,1,1,1,6,1,1,16,1,1,1,1,1
8,6,1,1,1,1,1,1,1,2,1,5,0,0,0,0,0
9,1,3,2,1,1,1,1,1,1,1,0,0,0,0,0,0


## tensorflow 기반의 정수인코딩/패딩 처리

In [13]:
# %pip install tensorflow

In [14]:
# 토큰화/정수 변환
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words = vocab_size+1,oov_token='OOV')

# 단어 사전 구축
tokenizer.fit_on_texts(preprocessed_sentences)

print(tokenizer.word_index)

{'OOV': 1, 'prince': 2, 'little': 3, 'pilot': 4, 'rose': 5, 'fox': 6, 'young': 7, 'planet': 8, 'earth': 9, 'story': 10, 'plane': 11, 'meets': 12, 'asteroid': 13, 'lessons': 14, 'love': 15, 'importance': 16, 'written': 17, 'antoine': 18, 'saint-exupéry': 19, 'poetic': 20, 'tale': 21, 'travels': 22, 'home': 23, 'begins': 24, 'stranded': 25, 'sahara': 26, 'desert': 27, 'crashes': 28, 'trying': 29, 'fix': 30, 'mysterious': 31, 'boy': 32, 'comes': 33, 'small': 34, 'called': 35, 'b-612': 36, 'lives': 37, 'alone': 38, 'loves': 39, 'deeply': 40, 'recounts': 41, 'journey': 42, 'describing': 43, 'visits': 44, 'several': 45, 'planets': 46, 'inhabited': 47, 'different': 48, 'character': 49, 'king': 50, 'vain': 51, 'man': 52, 'drunkard': 53, 'businessman': 54, 'geographer': 55, 'encounters': 56, 'learns': 57, 'valuable': 58, 'responsibility': 59, 'nature': 60, 'adult': 61, 'behavior': 62, 'various': 63, 'creatures': 64, 'including': 65, 'teaches': 66, 'relationships': 67, 'taming': 68, 'means': 69,

In [16]:
# 인덱스 변환
encoded_sentences = tokenizer.texts_to_sequences(preprocessed_sentences)
encoded_sentences

[[3, 2, 17, 18, 19, 20, 21, 7, 2, 22, 23, 8, 9],
 [10, 24, 4, 25, 26, 27, 11, 28],
 [29, 30, 11, 12, 1, 7, 1, 3, 2],
 [3, 2, 1, 1, 13, 1, 1, 1, 1, 5, 1, 1],
 [1, 1, 4, 1, 1, 1, 1],
 [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6],
 [1, 2, 1, 1, 14, 15, 1, 1, 1, 1],
 [9, 3, 2, 12, 1, 1, 1, 6, 1, 1, 16, 1, 1, 1, 1, 1],
 [6, 1, 1, 1, 1, 1, 1, 1, 2, 1, 5],
 [1, 3, 2, 1, 1, 1, 1, 1, 1, 1],
 [1, 1, 4, 1, 1, 13, 1, 5],
 [10, 1, 4, 1, 14, 1, 3, 2, 1, 1, 1],
 [1, 1, 1, 1, 1, 1, 15, 1, 16, 1, 1, 1, 1]]

In [18]:
# 패딩 처리
from tensorflow.keras.preprocessing.sequence import pad_sequences

# - padding: pre(기본값) | past
padded_sentences = pad_sequences(encoded_sentences, padding='post')
padded_sentences

array([[ 3,  2, 17, 18, 19, 20, 21,  7,  2, 22, 23,  8,  9,  0,  0,  0],
       [10, 24,  4, 25, 26, 27, 11, 28,  0,  0,  0,  0,  0,  0,  0,  0],
       [29, 30, 11, 12,  1,  7,  1,  3,  2,  0,  0,  0,  0,  0,  0,  0],
       [ 3,  2,  1,  1, 13,  1,  1,  1,  1,  5,  1,  1,  0,  0,  0,  0],
       [ 1,  1,  4,  1,  1,  1,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 8,  1,  1,  1,  1,  1,  1,  1,  1,  1,  6,  0,  0,  0,  0,  0],
       [ 1,  2,  1,  1, 14, 15,  1,  1,  1,  1,  0,  0,  0,  0,  0,  0],
       [ 9,  3,  2, 12,  1,  1,  1,  6,  1,  1, 16,  1,  1,  1,  1,  1],
       [ 6,  1,  1,  1,  1,  1,  1,  1,  2,  1,  5,  0,  0,  0,  0,  0],
       [ 1,  3,  2,  1,  1,  1,  1,  1,  1,  1,  0,  0,  0,  0,  0,  0],
       [ 1,  1,  4,  1,  1, 13,  1,  5,  0,  0,  0,  0,  0,  0,  0,  0],
       [10,  1,  4,  1, 14,  1,  3,  2,  1,  1,  1,  0,  0,  0,  0,  0],
       [ 1,  1,  1,  1,  1,  1, 15,  1, 16,  1,  1,  1,  1,  0,  0,  0]],
      dtype=int32)

In [20]:
padded_sentences = pad_sequences(
    encoded_sentences,
    maxlen=13,          # 최대 길이
    padding='post',
    truncating='post'       # 토큰을 자를 위치 : 기본 값 pre
    )
padded_sentences

array([[ 3,  2, 17, 18, 19, 20, 21,  7,  2, 22, 23,  8,  9],
       [10, 24,  4, 25, 26, 27, 11, 28,  0,  0,  0,  0,  0],
       [29, 30, 11, 12,  1,  7,  1,  3,  2,  0,  0,  0,  0],
       [ 3,  2,  1,  1, 13,  1,  1,  1,  1,  5,  1,  1,  0],
       [ 1,  1,  4,  1,  1,  1,  1,  0,  0,  0,  0,  0,  0],
       [ 8,  1,  1,  1,  1,  1,  1,  1,  1,  1,  6,  0,  0],
       [ 1,  2,  1,  1, 14, 15,  1,  1,  1,  1,  0,  0,  0],
       [ 9,  3,  2, 12,  1,  1,  1,  6,  1,  1, 16,  1,  1],
       [ 6,  1,  1,  1,  1,  1,  1,  1,  2,  1,  5,  0,  0],
       [ 1,  3,  2,  1,  1,  1,  1,  1,  1,  1,  0,  0,  0],
       [ 1,  1,  4,  1,  1, 13,  1,  5,  0,  0,  0,  0,  0],
       [10,  1,  4,  1, 14,  1,  3,  2,  1,  1,  1,  0,  0],
       [ 1,  1,  1,  1,  1,  1, 15,  1, 16,  1,  1,  1,  1]], dtype=int32)

In [34]:
# 실습 해보기
sentences: list[str] = [
    "The barber was a person.",
    "The barber was a good person.",
    "The barber was a huge person.",
    "He knew a secret.",
    "The secret was kept as a huge secret.",
    "It was a huge secret.",
    "The barber knew the huge secret.",
    "The barber kept his word.",
    "The barber kept the secret.",
    "Keeping the huge secret was driving the barber crazy.",
    "The barber went to the huge mountain."
]

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
# 전처리 함수 별도로 정의
# - 소문자 변환, 토큰화, 불용어 제거, 알파벳만, 단어 길이 3 이상만 사용
def preprocessing(text):
    preprocessed_sentences = []
    en_stopwords = stopwords.words('english')

    for sentence in text:
        sentence = sentence.lower()
        tokens = word_tokenize(sentence)
        # 불용어, 단어 길이가 3개 미만 필터링
        tokens = [token for token in tokens if token.isalpha() and token not in en_stopwords and len(token) > 2]
        preprocessed_sentences.append(tokens)
    return preprocessed_sentences

preprocessed_sentences = preprocessing(sentences)
print(preprocessed_sentences)

# 단어별 정수 인덱스 부여
tokenizer = Tokenizer(num_words = vocab_size+1,oov_token='OOV')
tokenizer.fit_on_texts(preprocessed_sentences)
print(tokenizer.word_index)

# 문장을 정수 시퀀스로 변환
encoded_sentences = tokenizer.texts_to_sequences(preprocessed_sentences)
print(encoded_sentences)

# 패딩 적용
padded_sentences = pad_sequences(
    encoded_sentences,
    maxlen=6,          # 최대 길이
    padding='post',
    truncating='post'       # 토큰을 자를 위치 : 기본 값 pre
    )
print(padded_sentences)

[['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'knew', 'huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]
{'OOV': 1, 'barber': 2, 'secret': 3, 'huge': 4, 'person': 5, 'kept': 6, 'knew': 7, 'good': 8, 'word': 9, 'keeping': 10, 'driving': 11, 'crazy': 12, 'went': 13, 'mountain': 14}
[[2, 5], [2, 8, 5], [2, 4, 5], [7, 3], [3, 6, 4, 3], [4, 3], [2, 7, 4, 3], [2, 6, 9], [2, 6, 3], [10, 4, 3, 11, 2, 12], [2, 13, 4, 14]]
[[ 2  5  0  0  0  0]
 [ 2  8  5  0  0  0]
 [ 2  4  5  0  0  0]
 [ 7  3  0  0  0  0]
 [ 3  6  4  3  0  0]
 [ 4  3  0  0  0  0]
 [ 2  7  4  3  0  0]
 [ 2  6  9  0  0  0]
 [ 2  6  3  0  0  0]
 [10  4  3 11  2 12]
 [ 2 13  4 14  0  0]]


In [42]:
# 전처리 함수 별도로 정의해서 문장 전처리
# - 소문자 변환, 토큰화, 불용어 제거, 알파벳만, 단어 길이 3 이상만 사용

en_stopwords = set(stopwords.words('english'))  # 영어 불용어

def preprocess_text(text):
    # 1. 소문자 변환
    text = text.lower()
    
    # 2. 토큰화
    tokens = word_tokenize(text)
    
    # 3. 불용어 제거 + 알파벳만 남기기 + 단어 길이 3 이상만 사용
    tokens = [
        token for token in tokens
        if token.isalpha() and token not in en_stopwords and len(token) >= 3
    ]
    
    return tokens

# 문장 전처리
processed_sentences = [preprocess_text(sentence) for sentence in sentences]

print("processed_sentences")
for original, processed in zip(sentences, processed_sentences):
    print(f"{original} -> {processed}")
print()

# 단어별 정수 인덱스 부여
tokenizer = Tokenizer(oov_token="OOV")
tokenizer.fit_on_texts(processed_sentences)

word_index = tokenizer.word_index   # 단어 사전 확인
print("word_index")
print(word_index)
print()

# 문장을 정수 시퀀스로 변환
sequences = tokenizer.texts_to_sequences(processed_sentences)
print("sequences")
for processed, seq in zip(processed_sentences, sequences):
    print(f"{processed} -> {seq}")
print()

max_len = max(len(seq) for seq in sequences)    # 최대 길이 확인
print("max_len:", max_len)
print()

# 패딩 적용
padded_sequences = pad_sequences(
    sequences,
    maxlen=max_len,
    padding="post",
    truncating="post"
)

print("padded_sequences")
print(padded_sequences)
print()

processed_sentences
The barber was a person. -> ['barber', 'person']
The barber was a good person. -> ['barber', 'good', 'person']
The barber was a huge person. -> ['barber', 'huge', 'person']
He knew a secret. -> ['knew', 'secret']
The secret was kept as a huge secret. -> ['secret', 'kept', 'huge', 'secret']
It was a huge secret. -> ['huge', 'secret']
The barber knew the huge secret. -> ['barber', 'knew', 'huge', 'secret']
The barber kept his word. -> ['barber', 'kept', 'word']
The barber kept the secret. -> ['barber', 'kept', 'secret']
Keeping the huge secret was driving the barber crazy. -> ['keeping', 'huge', 'secret', 'driving', 'barber', 'crazy']
The barber went to the huge mountain. -> ['barber', 'went', 'huge', 'mountain']

word_index
{'OOV': 1, 'barber': 2, 'secret': 3, 'huge': 4, 'person': 5, 'kept': 6, 'knew': 7, 'good': 8, 'word': 9, 'keeping': 10, 'driving': 11, 'crazy': 12, 'went': 13, 'mountain': 14}

sequences
['barber', 'person'] -> [2, 5]
['barber', 'good', 'person'] 